<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 07</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Normalization and Case Study</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:

* Normal Forms: 1NF, 2NF, 3NF, & BCNF
* Denormalization: When and why to break the rules.
* **case Study**:
   - Normalizations
   - Basic Query
   - Join
   - Aggregations

# Database Design: Redundancy, Anomalies, and Functional Dependencies

Efficient database design is critical for maintaining data integrity, especially in research environments where data accuracy and consistency are paramount. This explanation covers the pitfalls of poor schema design and the formal logic used to correct them.

## 1. Data Redundancy and Anomalies in Research Data

In a relational database, **data redundancy** occurs when the same piece of data is stored in multiple places unnecessarily. While redundant backups are good, redundancy within a schema leads to significant operational risks known as **anomalies**.

### 1.1 The Impact of Redundancy
* **Storage Inefficiency:** Large-scale research datasets (e.g., genomic data or longitudinal surveys) can become prohibitively expensive to store if duplicate information is recorded across millions of rows.
* **Data Inconsistency:** The primary danger is that one instance of a data point may be updated while others are not, leading to conflicting "truths" within the same dataset.

### 1.2 Database Anomalies
Anomalies are problems that arise during the manipulation of a poorly structured table. They are generally categorized into three types:

| Anomaly Type | Description | Research Example |
| :--- | :--- | :--- |
| **Update Anomaly** | Occurs when a change to a single data value requires multiple rows to be updated. | If a Lab Location changes, and it is stored in every row of an "Experiments" table, failing to update every row leads to inconsistent records. |
| **Insertion Anomaly** | Occurs when certain data cannot be recorded unless other, unrelated data is also provided. | You cannot record a new Researcher's profile until they are assigned to an active Experiment, because the primary key requires an Experiment ID. |
| **Deletion Anomaly** | Occurs when deleting one piece of information unintentionally results in the loss of other unrelated data. | Deleting the last Experiment entry for a specific Lab might accidentally delete all contact information for that Lab. |




## 2. Functional Dependencies (FD)

**Functional Dependency** is a constraint between two sets of attributes in a relation. It is the fundamental building block of **Normalization**, the process used to eliminate redundancy and anomalies.

### 2.1 Formal Definition
If $R$ is a relation and $X$ and $Y$ are subsets of attributes in $R$, we say $Y$ is **functionally dependent** on $X$ (denoted as $X \to Y$) if and only if each value of $X$ is associated with exactly one value of $Y$.

In simpler terms, if you know the value of $X$, you can determine the value of $Y$.

> **Example:** In a research database, `ResearcherID \to ResearcherName`. 
> If `ResearcherID` is 101, the name will always be "Dr. Smith." You won't find `ResearcherID` 101 associated with "Dr. Jones."



### 2.2 Types of Functional Dependencies

To refine database structures, we identify specific categories of FDs:

#### A. Full Functional Dependency
An attribute is fully functionally dependent if it depends on the *entire* primary key, not just a part of it.
* **Example:** In a table where the key is `{Student_ID, Course_ID}`, the `Grade` is fully dependent because you need both the student and the course to determine the grade.

#### B. Partial Functional Dependency
This occurs when an attribute depends on only a portion of a composite primary key. This is a primary cause of redundancy.
* **Example:** In the same `{Student_ID, Course_ID}` table, `Student_Name` is only dependent on `Student_ID`. Storing it here creates partial dependency.

#### C. Transitive Dependency
This occurs when a non-key attribute depends on another non-key attribute, which in turn depends on the primary key ($A \to B$ and $B \to C$).
* **Example:** `Project_ID \to Department_ID` and `Department_ID \to Department_Head`. Here, `Department_Head` has a transitive dependency on `Project_ID`.

### 2.3 Role in Normalization
Functional dependencies allow us to decompose a single, bloated table into smaller, logically sound tables:
1.  **First Normal Form (1NF):** Remove multi-valued attributes.
2.  **Second Normal Form (2NF):** Remove **Partial Dependencies**.
3.  **Third Normal Form (3NF):** Remove **Transitive Dependencies**.

By identifying $X \to Y$ relationships, researchers ensure that each fact is stored exactly once, preserving the integrity of the scientific record.

<center><img alt="" src="images/deds-05/functional-dependency.png" style="height: 600px;"/></center>

# Second Normal Form (2NF)

We review the transition from First Normal Form (1NF) to **Second Normal Form (2NF)**. While 1NF ensures atomicity, it often retains structural flaws that lead to data redundancy. The objective of 2NF is to eliminate **Partial Functional Dependencies**.

## 1. Prerequisites and Initial State
To achieve 2NF, a relation must first satisfy all criteria for **1NF**. We begin with the following 1NF table from the `university_registry_db`, where the composite primary key is defined as `{StudentID, CourseID}`.

### Table: `Student_Course_Registry_1NF`
| StudentID | CourseID | StudentName | Major | CourseTitle | Grade |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 2021001 | MATH101 | Alice Smith | Mathematics | Algebra | A |
| 2021001 | PHYS202 | Alice Smith | Mathematics | Mechanics | B |
| 2021002 | CS101 | Bob Johnson | Data Science | Intro to CS | A |

> **Reviewer's Note:** Observe that the primary key is a composite of two attributes. Any attribute that depends on only one part of this key is considered a "partial dependency."

## 2. Identifying Partial Functional Dependencies

In 2NF, every non-prime attribute (attributes not part of any candidate key) must be **fully functionally dependent** on the entire primary key. We identify the following dependencies in our table:

1.  **Full Dependency:** `{StudentID, CourseID} $\to$ Grade` (A grade requires both the student and the specific course).
2.  **Partial Dependency A:** `StudentID $\to$ {StudentName, Major}` (The student's name does not change based on the course).
3.  **Partial Dependency B:** `CourseID $\to$ CourseTitle` (The course title is independent of which student is enrolled).

## 3. The Decomposition Methodology
The reviewer suggests a systematic decomposition to resolve these dependencies. We must separate the partially dependent attributes into new relations where they depend on a **whole** primary key.

### Step 1: Isolate Student Data
Create a `Students` table where `StudentID` is the unique primary key. This removes the need to repeat the student's name for every course they take.

### Step 2: Isolate Course Data
Create a `Courses` table where `CourseID` is the primary key. This prevents repeating the course title for every enrolled student.

### Step 3: Maintain Relationships
Retain a "link" or "junction" table to store attributes that truly depend on both keys (like the grade).

## 4. Resulting 2NF Schema

The original bloated table is now decomposed into three logically sound relations.

### Table: `Students`
| StudentID | StudentName | Major |
| :--- | :--- | :--- |
| 2021001 | Alice Smith | Mathematics |
| 2021002 | Bob Johnson | Data Science |

### Table: `Courses`
| CourseID | CourseTitle |
| :--- | :--- |
| MATH101 | Algebra |
| PHYS202 | Mechanics |
| CS101 | Intro to CS |

### Table: `Enrollments`
| StudentID | CourseID | Grade |
| :--- | :--- | :--- |
| 2021001 | MATH101 | A |
| 2021001 | PHYS202 | B |
| 2021002 | CS101 | A |

## 5. Summary of 2NF Compliance
A relation is in **Second Normal Form** if:
1.  It is already in **1NF**.
2.  It contains **no partial functional dependencies**.
3.  All non-key attributes are functionally dependent on the **entire** primary key.

By achieving 2NF, we have successfully mitigated the **Update Anomaly**. For instance, if a student changes their major, we only need to update a single row in the `Students` table.

<center><img alt="" src="images/deds-05/2NF-db.png" style="height: 600px;"/></center>

# Analysis of Third Normal Form (3NF)

The following discourse evaluates the transition from Second Normal Form (2NF) to **Third Normal Form (3NF)**. This phase of normalization is essential for eliminating **transitive functional dependencies**, ensuring that every non-key attribute depends strictly on the primary key.

## 1. Prerequisites and Initial State

To qualify for 3NF, a relation must first satisfy all criteria for **2NF**. We examine a single table from the `university_registry_db` that, while in 2NF, still harbors structural redundancies.

### Table: `Student_Department_2NF`

| StudentID (PK) | StudentName | DeptID | DeptName | DeptHead |
| :--- | :--- | :--- | :--- | :--- |
| 2021001 | Alice Smith | D01 | Mathematics | Dr. Aris |
| 2021002 | Bob Johnson | D02 | Data Science | Dr. Budi |
| 2021003 | Charlie Davis | D01 | Mathematics | Dr. Aris |

### Observations:
In this schema, `StudentID` is the primary key. However, we observe a multi-step dependency chain:
1.  **Direct Dependency:** `StudentID` determines `DeptID`.
2.  **Transitive Dependency:** `DeptID` determines `DeptName` and `DeptHead`.

Consequently, `DeptName` is transitively dependent on `StudentID` via `DeptID`. This creates a redundancy where department details are repeated for every student in that department.

## 2. The Decomposition Methodology

The reviewer suggests a systematic decomposition to remove the intermediate determinant. This process isolates the transitive attributes into a distinct relation.

### Step 1: Identify the Determinant
Identify the non-key attribute that acts as a determinant for other non-key attributes. In our case, `DeptID` determines `DeptName` and `DeptHead`.

### Step 2: Extract to a New Relation
Remove the transitively dependent columns (`DeptName`, `DeptHead`) from the original table and place them into a new `Departments` table. In this new table, `DeptID` serves as the primary key.

### Step 3: Establish a Logical Link
Retain `DeptID` in the original table. It now functions as a **foreign key**, linking the student record to the correct department data without duplicating descriptive text.

## 3. Resulting 3NF Schema
The decomposition results in two streamlined, logically sound tables.

### Table: `Students`
| StudentID (PK) | StudentName | DeptID (FK) |
| :--- | :--- | :--- |
| 2021001 | Alice Smith | D01 |
| 2021002 | Bob Johnson | D02 |
| 2021003 | Charlie Davis | D01 |

### Table: `Departments`
| DeptID (PK) | DeptName | DeptHead |
| :--- | :--- | :--- |
| D01 | Mathematics | Dr. Aris |
| D02 | Data Science | Dr. Budi |

## 4. Summary of 3NF Compliance

A relation is in **Third Normal Form** if:
1.  It satisfies all requirements of **2NF**.
2.  It contains **no transitive dependencies**.
3.  Every non-key attribute is dependent **only** on the primary key.

In academic terms, this state ensures that each attribute provides a fact about "the key, the whole key, and nothing but the key."

## 5. Normalization Beyond 3NF

While 3NF is typically sufficient for most research and enterprise applications, advanced normalization forms exist to handle more complex scenarios. These include **Boyce-Codd Normal Form (BCNF)** for overlapping candidate keys, as well as **4NF** and **5NF** for managing multi-valued and join dependencies.

<center><img alt="" src="images/deds-05/3NF-db.png" style="height: 600px;"/></center>

# Denormalization: Strategic Redundancy in Database Management

While normalization seeks to eliminate redundancy and preserve integrity, **denormalization** is a strategic architectural choice to reintroduce redundancy. This technique is employed when the computational cost of data retrieval outweighs the benefits of a normalized schema.

## 1. The Rationale: Why Break the Rules?

The reviewer notes that normalization, while theoretically sound, often imposes significant overhead during query execution. The primary motivations for denormalization include:

* **Performance Optimization:** In systems with high-frequency read operations, the cost of performing multiple $R_1 \Join R_2 \Join R_3$ (JOIN) operations can lead to unacceptable latency. Denormalization reduces the number of joins required.
* **Query Simplification:** Complex queries involving many tables are difficult to write and maintain. Merging related attributes into a single relation simplifies the application logic.
* **Reporting and Analytics:** In Online Analytical Processing (OLAP) environments, historical data is often denormalized into "Star Schemas" to facilitate rapid aggregation and trend analysis.

## 2. Triggering Conditions: When to Denormalize

Denormalization is not a license for poor design but a deliberate optimization. It should be considered under the following conditions:

1.  **High Read-to-Write Ratio:** When data is read thousands of times for every single update, the cost of redundancy (increased update time) is negligible compared to the read savings.
2.  **Strict Latency Requirements:** If a system must deliver responses in milliseconds and the JOIN overhead exceeds this threshold.
3.  **Derived Data Requirements:** When a specific value (e.g., a total sum or average) is frequently requested but expensive to calculate on the fly.
4.  
## 3. Case Study: The Research Publication Registry

Consider a normalized system tracking research papers and their departments.

**Normalized State (3NF):**
* `Papers` table (PaperID, Title, DeptID)
* `Departments` table (DeptID, DeptName)

To display a list of paper titles with their department names, the system must perform a JOIN for every request.

**Denormalized State:**
* `Papers_Denormalized` table (PaperID, Title, DeptID, **DeptName**)

By storing the `DeptName` directly in the `Papers` table, the JOIN is eliminated. The reviewer cautions, however, that if a department name changes, the system must now update multiple rows instead of one.

## 4. Implementation via Python

In modern data science workflows, denormalization is frequently performed during the data preparation phase using the `pandas` library.

```python
import pandas as pd

# 1. Normalized Data Sources
papers_df = pd.DataFrame({
    'PaperID': [101, 102, 103],
    'Title': ['Green AI', 'Big Data Ethics', 'EcoCluster'],
    'DeptID': ['D1', 'D1', 'D2']
})

depts_df = pd.DataFrame({
    'DeptID': ['D1', 'D2'],
    'DeptName': ['Computer Science', 'Environmental Science']
})

# 2. Denormalization Process (The "Break")
# We merge the tables to create a flat, redundant structure for faster access
denormalized_df = pd.merge(papers_df, depts_df, on='DeptID', how='left')

# 3. View the Resulting Redundancy
print("Denormalized Research Registry:")
print(denormalized_df)

# The 'DeptName' is now repeated, allowing for O(1) access per row 
# without further lookups or joins.
```

## 5. Summary Conclusion

Denormalization is a trade-off: you sacrifice **storage efficiency** and **write simplicity** to gain **read speed**. The reviewer suggests that for large-scale research infrastructures, such as those used in Green AI or high-frequency data streaming, denormalization is often a necessary departure from traditional relational theory.

<center><img alt="" src="images/deds-05/denormalization-db.png" style="height: 600px;"/></center>

<center><h2><strong><font color="green">Start designing database schema for your thesis project</font></strong></h2></center>

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

## Next Week's Discussions

* Entity-Relationship Diagrams (ERD).
* Data Acquisition II – Web Scraping & Crawling
   - Legal & Ethics of Scraping (robots.txt, Terms of Service).
   - Data Streaming
   - Normalization Assignment

<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-database-primary-keys.jpeg" style="height: 600px;"/></center>